# AEF trading simulation: a first submission in Python

This notebook goes through one decision: see what you may trade, download prices, build a portfolio, check it and submit it. Run it cell by cell; nothing is sent until the `submit()` cell.

## Setup

Install once, with uv (`uv add ...` in your project) or pip:

```sh
uv add "git+https://github.com/advanced-empirical-finance/aefsim#subdirectory=python" tidyfinance
pip install "git+https://github.com/advanced-empirical-finance/aefsim#subdirectory=python" tidyfinance
```

Your team's token goes into the environment variable `AEF_TOKEN` (for example `export AEF_TOKEN=aef_...` in your shell, or a `.env` file you never commit). Never paste it into a notebook you share.

In [ ]:
import datetime as dt

import aefsim
import polars as pl
import tidyfinance as tf

## What can we trade, and under which rules?

In [ ]:
universe = aefsim.get_universe()
as_of = universe["as_of"].max()  # the latest close
print(as_of)
universe.group_by("eligible", "sp500").len()

Only `eligible` stocks can be bought: CRSP-style common stocks on NYSE, NYSE American and Nasdaq that closed at $5 or more. A stock you hold that drops below $5 may be held or reduced, not increased.

In [ ]:
aefsim.get_parameters()  # costs, shorting, leverage, position cap; applies_next marks the current set

In [ ]:
aefsim.get_calendar().filter(pl.col("is_next"))  # the next fill and its deadline, 09:00 New York time

## Prices

The simulation does not serve prices. tidyfinance downloads them from Yahoo Finance, the same source the simulation fills at. Like aefsim, it returns polars DataFrames.

In [ ]:
candidates = universe.filter(pl.col("eligible") & pl.col("sp500"))["ticker"].to_list()
prices = tf.download_data(
    domain="stock_prices",
    symbols=candidates,
    start_date=str(as_of - dt.timedelta(days=365)),
    end_date=str(as_of),
)

## A simple rule

As an example (not a recommendation): the 20 S&P 500 stocks with the lowest volatility over the past year, equal weight.

In [ ]:
volatility = (
    prices.sort("symbol", "date")
    .with_columns(ret=pl.col("adjusted_close").pct_change().over("symbol"))
    .group_by("symbol")
    .agg(vol=pl.col("ret").std() * 252 ** 0.5, days=pl.len())
    .filter(pl.col("days") >= 200)
    .sort("vol")
    .head(20)
)
weights = {symbol: 1 / volatility.height for symbol in volatility["symbol"]}
weights

Weights are fractions of your portfolio's value. Tickers you leave out get zero, so this replaces your whole portfolio; whatever does not add up to one stays in cash.

## Check, then submit

In [ ]:
check = aefsim.validate(weights, as_of=as_of)
check

`validate()` runs every check the simulation runs and stores nothing. When it says OK, submit before the deadline. The rationale is your decision log: write down what you expect and why.

In [ ]:
result = aefsim.submit(
    weights,
    as_of=as_of,
    rationale="Low-volatility tilt: equal weight in the 20 least volatile S&P 500 stocks over the past year.",
)
result

Changed your mind? Submit again (the last accepted submission before the deadline is filled), or `aefsim.cancel(result.submission_id)`.

## The next morning

Fills happen at the US open of the fill date; results appear after the nightly update.

In [ ]:
aefsim.get_submissions()

In [ ]:
portfolio = aefsim.get_portfolio()
portfolio["positions"]

In [ ]:
aefsim.get_performance()["summary"]

In [ ]:
aefsim.get_leaderboard()